# Principal Component Analysis

This notebook implements Principal Component Analysis using a scratch implementation and compares it to `scikit-learn`.
The analysis is inspired by ([A Tutorial on Principal Component Analysis (Shlens, 2014)](https://arxiv.org/pdf/1404.1100)), including geometric intuition and variance explained.


## Data generation

Generate a synthetic dataset with a clear linear relationship across 3 dimensions.


In [1]:
import sys
import pathlib
from time import perf_counter

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.decomposition import PCA as SKPCA

# Ensure the local scratch implementation is importable from the notebook path
sys.path.append(str(pathlib.Path.cwd() / 'papers' / 'pca'))
from pca_scratch import PCASVD, PCAEigen

np.random.seed(0)
n = 200
x = np.random.normal(size=n)
y = 2 * x + np.random.normal(scale=0.6, size=n)
z = -x + 0.5 * y + np.random.normal(scale=0.4, size=n)
X = np.column_stack([x, y, z])
df = pd.DataFrame(X, columns=['x', 'y', 'z'])
df.head()


,x,y,z
0,1.764052,3.306596,-0.350216
1,0.400157,0.656687,-0.518173
2,0.978738,2.617272,0.636563
3,2.240893,4.874945,0.339096
4,1.867558,4.119195,-0.515376


In [2]:
fig = px.scatter_3d(df, x='x', y='y', z='z', title='Original 3D data distribution', width=800, height=600)
fig.update_traces(marker=dict(size=4))
fig.show()


## PCA implementation from scratch (SVD)

Fit the scratch PCA implementation using singular value decomposition and inspect the principal components and explained variance ratios.


In [3]:
pca_svd = PCASVD(n_components=3, whiten=False)
X_svd = pca_svd.fit_transform(X)
explained_svd = pca_svd.explained_variance_ratio_
components_svd = pca_svd.components_
print('Scratch PCA (SVD) explained variance ratio:', np.round(explained_svd, 4))
print('Scratch PCA (SVD) components shape:', components_svd.shape)

Scratch PCA (SVD) explained variance ratio: [0.9426 0.0523 0.0051]
Scratch PCA (SVD) components shape: (3, 3)


## PCA implementation from scratch (Eigen decomposition)

Fit the scratch PCA implementation using eigen decomposition and compare its output to the SVD-based implementation.


In [4]:
pca_eigen = PCAEigen(n_components=3, whiten=False)
X_eigen = pca_eigen.fit_transform(X)
explained_eigen = pca_eigen.explained_variance_ratio_
components_eigen = pca_eigen.components_
print('Scratch PCA (eigen) explained variance ratio:', np.round(explained_eigen, 4))
print('Scratch PCA (eigen) components shape:', components_eigen.shape)
print('Max abs difference between SVD and eigen component magnitudes:',
      np.max(np.abs(np.abs(components_svd) - np.abs(components_eigen))))

Scratch PCA (eigen) explained variance ratio: [0.9426 0.0523 0.0051]
Scratch PCA (eigen) components shape: (3, 3)
Max abs difference between SVD and eigen component magnitudes: 1.8318679906315083e-15


## PCA with scikit-learn

Fit `sklearn` PCA on the same data and compare explained variance ratios.


In [5]:
pca_sklearn = SKPCA(n_components=3, whiten=False)
X_sklearn = pca_sklearn.fit_transform(X)
explained_sklearn = pca_sklearn.explained_variance_ratio_
components_sklearn = pca_sklearn.components_
print('scikit-learn PCA explained variance ratio:', np.round(explained_sklearn, 4))

scikit-learn PCA explained variance ratio: [0.9426 0.0523 0.0051]


## Visualization

Compare the original 3D data to the PCA projection from the SVD-based scratch implementation.


In [6]:
X_centered = X - X.mean(axis=0)
mean = X.mean(axis=0)
pc1 = components_svd[0]
pc2 = components_svd[1]
plane_scale = np.max(np.std(X_centered, axis=0)) * 2.5
u = np.linspace(-plane_scale, plane_scale, 12)
v = np.linspace(-plane_scale, plane_scale, 12)
U, V = np.meshgrid(u, v)
plane = mean[:, None, None] + pc1[:, None, None] * U + pc2[:, None, None] * V
plane_pca_x, plane_pca_y = np.meshgrid(u, v)
plane_pca_z = np.zeros_like(plane_pca_x)

fig = make_subplots(
    rows=1, cols=2,
    specs=[[{'type': 'scene'}, {'type': 'scene'}]],
    subplot_titles=(
        'Original 3D data + PCA plane',
        'PCA projection on first two components'
    ),
)

fig.add_trace(
    go.Surface(
        x=plane[0],
        y=plane[1],
        z=plane[2],
        opacity=0.35,
        showscale=False,
        colorscale='Blues',
        name='PCA plane',
    ),
    row=1, col=1,
)
fig.add_trace(
    go.Scatter3d(
        x=X[:, 0],
        y=X[:, 1],
        z=X[:, 2],
        mode='markers',
        marker=dict(size=4, color=x, colorscale='Viridis', opacity=0.8),
        name='Original data',
    ),
    row=1, col=1,
)
fig.add_trace(
    go.Surface(
        x=plane_pca_x,
        y=plane_pca_y,
        z=plane_pca_z,
        opacity=0.25,
        showscale=False,
        colorscale='Blues',
        name='PCA plane',
    ),
    row=1, col=2,
)
fig.add_trace(
    go.Scatter3d(
        x=X_svd[:, 0],
        y=X_svd[:, 1],
        z=np.zeros(n),
        mode='markers',
        marker=dict(size=4, color=X_svd[:, 0], colorscale='Plasma', opacity=0.8),
        name='PCA result',
    ),
    row=1, col=2,
)

fig.update_layout(
    title='PCA via scratch implementation (SVD)',
    height=650,
    width=1200,
    showlegend=False,
)
fig.show()

## Explained Variance Comparison

Compare the explained variance ratios from SVD scratch PCA, eigen scratch PCA, and scikit-learn PCA.


In [7]:
variance_df = pd.DataFrame({
    'component': ['PC1', 'PC2', 'PC3'],
    'svd_ratio': explained_svd,
    'eigen_ratio': explained_eigen,
    'sklearn_ratio': explained_sklearn
})
fig = px.bar(variance_df, x='component', y=['svd_ratio', 'eigen_ratio', 'sklearn_ratio'],
             title='Explained variance ratio: SVD scratch, eigen scratch, and sklearn',
             labels={'value': 'Explained variance ratio', 'component': 'Principal component'},
             width=900, height=500)
fig.show()

## Performance Benchmarking

Measure runtime for repeated `fit_transform` calls for SVD scratch PCA, eigen scratch PCA, and scikit-learn PCA.


In [8]:
def measure_runtime(model_factory, X, repeats=30):
    times = []
    for _ in range(repeats):
        model = model_factory()
        start = perf_counter()
        model.fit_transform(X)
        times.append(perf_counter() - start)
    return np.mean(times), np.std(times)

svd_time, svd_std = measure_runtime(lambda: PCASVD(n_components=3), X, repeats=25)
eigen_time, eigen_std = measure_runtime(lambda: PCAEigen(n_components=3), X, repeats=25)
sklearn_time, sklearn_std = measure_runtime(lambda: SKPCA(n_components=3), X, repeats=25)
print(f'Scratch PCA (SVD) average runtime: {svd_time:.6f}s ± {svd_std:.6f}s (std dev)')
print(f'Scratch PCA (eigen) average runtime: {eigen_time:.6f}s ± {eigen_std:.6f}s (std dev)')
print(f'scikit-learn PCA average runtime: {sklearn_time:.6f}s ± {sklearn_std:.6f}s (std dev)')

Scratch PCA (SVD) average runtime: 0.000190s ± 0.000330s (std dev)
Scratch PCA (eigen) average runtime: 0.000073s ± 0.000012s (std dev)
scikit-learn PCA average runtime: 0.000437s ± 0.000080s (std dev)
